# Modelamiento predictivo de ventas semanales por SKU

Este notebook es el **laboratorio reproducible** del proyecto. La app Streamlit es la capa de presentación; aquí se documentan la selección de SKU, exploración, calibración, backtesting y entrenamiento final.

### Diseño definitivo

- Periodo estacional anual: **52 semanas**.
- Criterio de admisibilidad: **≥104 semanas = al menos 2 ciclos completos**.
- Selección: 10 SKU admisibles con mayor ingreso histórico.
- Variables futuras: precio, promoción, verano, invierno y feriados.
- Métricas: **WAPE** (principal) y **MAE** (secundaria).
- Backtest: **3 folds de 4 semanas** sobre los mismos periodos para todos los modelos.


## 1. Modelos candidatos

Se comparan cuatro especificaciones. No es necesario que sean matemáticamente independientes entre sí; lo importante es que sean especificaciones distintas y que reciban la misma evaluación fuera de muestra.

1. **Seasonal Naive**: $\hat y_t=y_{t-52}$.
2. **Fourier + Ridge**: tendencia + armónicos Fourier + exógenas, con regularización Ridge.
3. **SARIMAX estacional**: dependencia temporal estacional explícita a 52 semanas + exógenas. **No contiene Fourier**.
4. **Fourier + ARMA (DHR)**: regresión armónica dinámica; Fourier + exógenas y errores ARMA.

La presencia del cuarto modelo no invalida Fourier + Ridge: ambos comparten representación armónica, pero uno supone errores simples y el otro modela dependencia residual temporal.


## 2. Referencia científica y adaptación

**Referencia principal:** Kačmáry et al. (2024), *Forecast of sales of selected food products in retail using Fourier series analysis and non-linear regression*, Foresight, 26(3), 487–504, DOI 10.1108/FS-12-2022-0168.

**Referencia de apoyo:** de Castro Moraes, Yuan & Chew (2024), *Hybrid convolutional long short-term memory models for sales forecasting in retail*, Journal of Forecasting, 43(5), 1278–1293, DOI 10.1002/for.3073.

El notebook **adapta** ideas de forecasting de ventas y estacionalidad al dataset del curso. No se presenta como reproducción literal de los papers.


In [1]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

from forecast_core import (
    PERIOD, MIN_WEEKS, TOP_N, MODEL_NAMES,
    load_data, sku_summary, select_top_skus, aggregate_sku,
    decompose_stl, correlation_table, select_hyperparameters,
    rolling_backtest, fit_and_save_sku, load_saved_models,
    add_fourier_features, ridge_coefficients,
)

ROOT = Path.cwd()
DATA = ROOT / 'data' / 'weekly_df_final_for_modeling.xlsx'
ART = ROOT / 'artifacts'
MODELS = ROOT / 'models'
ART.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')


## 3. Carga y selección de SKU

`104` no es el periodo del modelo. El periodo es `52`; `104` es la cantidad de semanas que representa **dos ciclos anuales completos**.


In [2]:
df = load_data(DATA)
summary = sku_summary(df)
top10 = select_top_skus(summary, TOP_N)

print(f'Filas de la base: {len(df):,}')
print(f'SKU totales: {summary.sku.nunique()}')
print(f'SKU admisibles (≥{MIN_WEEKS} semanas): {summary.admissible.sum()}')
display(top10[['sku','common_weeks','seasonal_cycles','total_units','total_revenue']])


Filas de la base: 31,027
SKU totales: 30
SKU admisibles (≥104 semanas): 20


,sku,common_weeks,seasonal_cycles,total_units,total_revenue
0,YO-029,143,2.750,"173,238.000","909,515.728"
1,YO-005,142,2.731,"172,131.000","897,843.568"
2,YO-012,139,2.673,"165,720.000","873,428.091"
3,MI-026,149,2.865,"149,653.000","781,265.683"
4,RE-004,137,2.635,"146,675.000","773,954.405"
5,YO-014,138,2.654,"144,812.000","763,293.072"
6,YO-001,138,2.654,"142,143.000","752,976.888"
7,RE-007,135,2.596,"143,323.000","750,816.690"
8,RE-015,133,2.558,"141,280.000","745,039.065"
9,YO-009,122,2.346,"141,624.000","735,846.256"


In [3]:
plot_df = summary.sort_values('common_weeks', ascending=False).copy()
plot_df['estado'] = np.where(plot_df['admissible'], 'Admisible', 'No admisible')
fig = px.bar(
    plot_df, x='sku', y='common_weeks', color='estado',
    color_discrete_map={'Admisible':'#4E8F68','No admisible':'#D9534F'},
    labels={'sku':'SKU','common_weeks':'Semanas comunes'}
)
fig.add_hline(y=MIN_WEEKS, line_dash='dash', annotation_text='104 semanas = 2 ciclos')
fig.update_layout(title='Cobertura temporal por SKU', height=430)
fig.show()


In [4]:
pareto = summary.sort_values('total_revenue', ascending=False).copy()
pareto['cum_pct'] = 100 * pareto['total_revenue'].cumsum() / pareto['total_revenue'].sum()
fig = go.Figure()
fig.add_bar(x=pareto['sku'], y=pareto['total_revenue'], name='Ingreso histórico')
fig.add_scatter(x=pareto['sku'], y=pareto['cum_pct'], yaxis='y2', mode='lines+markers', name='% acumulado')
fig.update_layout(
    title='Pareto de ingresos históricos',
    yaxis={'title':'Ingresos'},
    yaxis2={'title':'% acumulado','overlaying':'y','side':'right','range':[0,105]},
    height=450
)
fig.show()


## 4. Exploración de un SKU

STL se usa de manera **descriptiva** para separar tendencia, estacionalidad y residuo. No decide automáticamente cuál modelo debe ganar.


In [5]:
example_sku = top10.iloc[0]['sku']
series = aggregate_sku(df, example_sku)
print(example_sku, '| semanas:', len(series), '| ciclos:', round(len(series)/PERIOD, 2))

fig = px.line(series, x='week', y='units_sold', title=f'Ventas semanales — {example_sku}')
fig.show()


YO-029 | semanas: 143 | ciclos: 2.75


In [6]:
stl = decompose_stl(series, PERIOD)
stl_df = pd.DataFrame({
    'week': stl.observed.index,
    'observed': stl.observed.values,
    'trend': stl.trend.values,
    'seasonal': stl.seasonal.values,
    'residual': stl.resid.values,
})
display(stl_df.tail())
fig = px.line(stl_df, x='week', y=['trend','seasonal','residual'], facet_row='variable',
              title=f'STL anual (periodo=52) — {example_sku}')
fig.update_yaxes(matches=None)
fig.update_layout(height=700)
fig.show()


,week,observed,trend,seasonal,residual
138,2024-11-18,882.000,"1,153.551",-271.638,0.087
139,2024-11-25,955.000,"1,152.656",-197.750,0.095
140,2024-12-02,799.000,"1,151.761",-352.863,0.102
141,2024-12-09,747.000,"1,150.866",-403.975,0.110
142,2024-12-16,791.000,"1,149.971",-359.088,0.117


In [7]:
corr = correlation_table(series)
display(corr[['variable','spearman_rho','p_value']])


,variable,spearman_rho,p_value
0,is_summer,0.382,0.000
1,is_winter,-0.267,0.001
2,promotion_flag,0.111,0.186
3,is_holiday_week,-0.068,0.419
4,price_unit,-0.052,0.538


## 5. Calibración previa al backtest externo

Para evitar escoger parámetros mirando el mismo periodo que luego usamos para reportar el desempeño, se separa el tiempo así:

`historia para calibrar parámetros → 12 semanas de backtest externo`

Dentro de la historia anterior se usa un bloque final de 4 semanas para escoger parámetros. Después esos parámetros quedan fijos durante los tres folds del backtest externo.

- Ridge: $lpha \in \{0.1,1,10,100\}$.
- DHR: AR(1), MA(1) o ARMA(1,1) en los errores.
- SARIMAX: especificaciones estacionales parsimoniosas a 52 semanas, sin Fourier.


In [8]:
example_params = select_hyperparameters(series)
example_params


{'fourier_k': 2,
 'ridge_alpha': 0.1,
 'dhr_order': (1, 0, 1),
 'sarimax_order': (1, 0, 0),
 'sarimax_seasonal_order': (1, 0, 0, 52),
 'tuning_train_weeks': 127,
 'tuning_validation_weeks': 4,
 'outer_test_weeks_reserved': 12}

## 6. Backtest de ejemplo

Los cuatro modelos pronostican **exactamente las mismas semanas**. WAPE determina el orden principal y MAE sirve como segunda métrica.


In [9]:
example_preds, example_metrics = rolling_backtest(series, params=example_params)
display(example_metrics)

fig = px.line(
    example_preds.sort_values('week'), x='week', y='prediction', color='model', markers=True,
    title=f'Predicciones de backtest — {example_sku}'
)
actual = example_preds[['week','actual']].drop_duplicates().sort_values('week')
fig.add_scatter(x=actual['week'], y=actual['actual'], mode='lines+markers', name='Real', line={'width':4})
fig.show()


,model,MAE,WAPE,rank
0,Fourier + ARMA (DHR),69.820,8.146,1
1,Fourier + Ridge,88.742,10.354,2
2,Seasonal Naive,95.000,11.084,3
3,SARIMAX estacional,101.655,11.861,4


## 7. Evaluación de los 10 SKU y guardado de resultados

Esta celda es la que tarda más: calibra parámetros, ejecuta el backtest común, reentrena cada modelo con toda la historia disponible y guarda los `.pkl` que luego reutiliza Streamlit.


> La versión entregada conserva los artefactos ya recalculados. `RETRAIN_ALL=False` permite abrir y ejecutar el notebook rápidamente; cambiarlo a `True` ejecuta nuevamente el entrenamiento completo de los 10 SKU.

In [10]:
RETRAIN_ALL = False  # Cambia a True si modificas la base o la metodología y quieres reentrenar todo.

if RETRAIN_ALL:
    # Limpiar resultados anteriores para no mezclar metodologías.
    for folder in [ART, MODELS]:
        for p in folder.iterdir():
            if p.is_file():
                p.unlink()
            elif p.is_dir():
                shutil.rmtree(p)

    summary.to_csv(ART / 'sku_summary.csv', index=False)
    top10.to_csv(ART / 'sku_selection.csv', index=False)

    all_metrics = []
    all_preds = []
    param_rows = []
    coef_rows = []
    registry_rows = []

    for i, sku in enumerate(top10['sku'], start=1):
        print(f'[{i}/{len(top10)}] {sku}')
        s = aggregate_sku(df, sku)
        params = select_hyperparameters(s)
        preds, metrics = rolling_backtest(s, params=params)
        preds['sku'] = sku
        metrics['sku'] = sku
        all_preds.append(preds)
        all_metrics.append(metrics)

        meta = fit_and_save_sku(s, sku, MODELS, metrics=metrics, params=params)
        models = load_saved_models(sku, MODELS)
        feature_names = add_fourier_features(s, s['week'].iloc[0], k=params['fourier_k']).columns.tolist()
        coef_rows.append(ridge_coefficients(models['ridge'], feature_names, sku))

        param_rows.append({
            'sku': sku,
            'fourier_k': params['fourier_k'],
            'ridge_alpha': params['ridge_alpha'],
            'dhr_order': str(tuple(params['dhr_order'])),
            'sarimax_order': str(tuple(params['sarimax_order'])),
            'sarimax_seasonal_order': str(tuple(params['sarimax_seasonal_order'])),
            'tuning_train_weeks': params['tuning_train_weeks'],
            'tuning_validation_weeks': params['tuning_validation_weeks'],
            'outer_test_weeks_reserved': params['outer_test_weeks_reserved'],
        })
        registry_rows.append({
            'sku': sku,
            'winner': meta.get('winner'),
            'winner_WAPE': meta.get('winner_WAPE'),
            'winner_MAE': meta.get('winner_MAE'),
            'n_weeks': meta['n_weeks'],
            'seasonal_cycles': meta['seasonal_cycles'],
            'seasonal_sarimax_converged': meta['seasonal_sarimax_converged'],
            'dhr_converged': meta['dhr_converged'],
        })

    metrics_all = pd.concat(all_metrics, ignore_index=True)
    preds_all = pd.concat(all_preds, ignore_index=True)
    params_all = pd.DataFrame(param_rows)
    coeff_all = pd.concat(coef_rows, ignore_index=True)
    registry = pd.DataFrame(registry_rows)

    metrics_all.to_csv(ART / 'model_metrics.csv', index=False)
    preds_all.to_csv(ART / 'backtest_predictions.csv', index=False)
    params_all.to_csv(ART / 'selected_parameters.csv', index=False)
    coeff_all.to_csv(ART / 'ridge_coefficients.csv', index=False)
    registry.to_csv(ART / 'model_registry.csv', index=False)
    registry.to_csv(ART / 'saved_models.csv', index=False)
    print('Reentrenamiento completo terminado.')
else:
    # Para una apertura rápida del notebook ejecutado, se reutilizan los resultados ya regenerados.
    # El bloque anterior conserva todo el procedimiento reproducible si se desea reentrenar.
    metrics_all = pd.read_csv(ART / 'model_metrics.csv')
    preds_all = pd.read_csv(ART / 'backtest_predictions.csv', parse_dates=['week'])
    params_all = pd.read_csv(ART / 'selected_parameters.csv')
    coeff_all = pd.read_csv(ART / 'ridge_coefficients.csv')
    registry = pd.read_csv(ART / 'model_registry.csv')
    print('Resultados actualizados cargados desde artifacts/. Para recalcular, usa RETRAIN_ALL = True.')


Resultados actualizados cargados desde artifacts/. Para recalcular, usa RETRAIN_ALL = True.


## 8. Resultados consolidados

No se impone un ganador global. Cada SKU conserva el modelo con menor WAPE en el backtest externo; esto permite que un SKU termine con Seasonal Naive mientras otro use Fourier + Ridge, SARIMAX estacional o DHR.


In [11]:
winners = (
    metrics_all.sort_values(['sku','WAPE','MAE'])
    .groupby('sku', as_index=False)
    .first()[['sku','model','WAPE','MAE']]
    .sort_values('WAPE')
)
display(winners)
print('\nDistribución de ganadores:')
display(winners['model'].value_counts().rename_axis('modelo').reset_index(name='n_sku'))

,sku,model,WAPE,MAE
4,YO-001,Fourier + Ridge,4.347,30.245
0,MI-026,Fourier + Ridge,5.311,46.142
2,RE-007,Seasonal Naive,7.213,80.167
7,YO-012,Fourier + Ridge,7.747,66.986
9,YO-029,Fourier + ARMA (DHR),8.146,69.820
8,YO-014,Fourier + Ridge,8.502,55.575
3,RE-015,SARIMAX estacional,8.514,90.595
1,RE-004,Fourier + Ridge,11.100,120.600
6,YO-009,Seasonal Naive,11.373,99.417
5,YO-005,Fourier + Ridge,11.461,98.796



Distribución de ganadores:


,modelo,n_sku
0,Fourier + Ridge,6
1,Seasonal Naive,2
2,Fourier + ARMA (DHR),1
3,SARIMAX estacional,1


In [12]:
pivot = metrics_all.pivot(index='sku', columns='model', values='WAPE')
display(pivot.round(2))


model,Fourier + ARMA (DHR),Fourier + Ridge,SARIMAX estacional,Seasonal Naive
sku,,,,
MI-026,7.230,5.310,8.660,11.550
RE-004,11.750,11.100,11.720,11.220
RE-007,8.740,7.300,7.290,7.210
RE-015,9.940,8.980,8.510,9.190
YO-001,11.070,4.350,17.850,11.860
YO-005,12.080,11.460,14.510,12.650
YO-009,14.310,13.170,16.340,11.370
YO-012,7.910,7.750,10.080,13.160
YO-014,11.030,8.500,13.860,11.210


## 9. Coeficientes de Fourier + Ridge

Los coeficientes se muestran sobre predictores **estandarizados**, por lo que sirven para comparar el peso predictivo relativo dentro del SKU. **No deben interpretarse como efectos causales.**


In [13]:
example_coef = coeff_all[coeff_all['sku'].eq(example_sku)].copy()
display(example_coef)
fig = px.bar(example_coef, x='standardized_coefficient', y='feature', orientation='h',
             title=f'Coeficientes Ridge estandarizados — {example_sku}')
fig.show()


,sku,feature,standardized_coefficient
99,YO-029,price_unit,8.710
100,YO-029,is_holiday_week,-10.612
101,YO-029,sin_2,-11.511
102,YO-029,cos_2,-24.211
103,YO-029,is_winter,-26.713
104,YO-029,is_summer,28.912
105,YO-029,promotion_flag,41.518
106,YO-029,t2,106.149
107,YO-029,sin_1,148.592
108,YO-029,t,-154.490


## 10. Qué pasa a Streamlit

La app **no vuelve a ejecutar todo este laboratorio** en cada apertura. Lee `artifacts/` para mostrar métricas y carga `models/` para generar el forecast. Los detalles técnicos quedan dentro de expanders para mantener la interfaz simple.

El forecast permite escoger manualmente cualquiera de los cuatro modelos o usar **“Mejor según WAPE”** para cargar el ganador del SKU.
